# Jumper — C sul gradino singolo da 15 cm con ripiano lungo (solo CPU, nessuna quota GPU)

Layout `curb`: un'alzata di 15 cm e 1,5 m di ripiano. Misura salto e tremolio con il filtro sui target, registra i 5 salti migliori (traiettorie per il render) e qualche video di controllo.

Input: Dataset `jumper-stairs-task` (v12+) e output del notebook `jumper-kaggle-stairs` (versione 9). Accelerator: None. Internet: On.

In [ ]:
# @title 1. Setup solo CPU (nessuna GPU, nessuna quota): clone, codice della task, venv
import glob, json, os, re, shutil, subprocess, sys, venv, zipfile
from pathlib import Path

REPOSITORY_URL = "https://github.com/KingKongRobotics/jumper.git"
SOURCE_REVISION = "b2e58669848a3ba4f209597cd6bebdfe9d0533a4"
TASK = "jumper.stairs_free"
ROW = "9"                                   # 15 cm
LAYOUT = "curb"                             # un'alzata, poi 1,5 m di ripiano
ALPHAS = "1,0.7,0.5"                        # filtro sui target: 1 = nessun filtro (0.3 e 0.2 rompono il salto)
NUM_ENVS, SECONDS = 16, 6.0
RECORD_ALPHAS = (0.5, 1.0)                  # traiettorie registrate, una cartella per alpha
RECORD_ENVS, RECORD_KEEP = 32, 5
VIDEOS = [  # (etichetta, alpha, rumore di osservazione)
    ("curb-alpha1-rumore", 1.0, True),
    ("curb-alpha0.7-rumore", 0.7, True),
    ("curb-alpha0.5-rumore", 0.5, True),
]
VIDEO_SECONDS = 6.0

WORK = Path("/kaggle/temp") if Path("/kaggle").is_dir() else Path("/tmp/jumper-curb")
OUT = Path("/kaggle/working/curb") if Path("/kaggle").is_dir() else Path.cwd() / "curb-out"
REPO = WORK / "jumper"
PYTHON = REPO / ".venv/bin/python"
for d in (WORK, OUT):
    d.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None, env=None, check=True):
    cmd = [str(c) for c in cmd]
    print("+", " ".join(cmd), flush=True)
    p = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-6000:], flush=True)
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)
    return p


if not REPO.exists():
    run(["git", "clone", REPOSITORY_URL, REPO])
run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO)
run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO)

# Codice della task, dal Dataset jumper-stairs-task (zip o cartella)
hits = glob.glob("/kaggle/input/**/tasks/jumper/stairs/env_cfg.py", recursive=True)
zips = [z for z in glob.glob("/kaggle/input/**/jumper-stairs-task*.zip", recursive=True)]
if zips:
    with zipfile.ZipFile(sorted(zips)[-1]) as z:
        assert not any(n.startswith("/") or ".." in Path(n).parts for n in z.namelist())
        z.extractall(REPO)
    print("Applicato", sorted(zips)[-1])
elif hits:
    root = Path(hits[0]).parents[3]
    for src in root.rglob("*"):
        if src.is_file():
            dst = REPO / src.relative_to(root)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    print("Copiato da", root)
else:
    raise FileNotFoundError("Add Input: il Dataset jumper-stairs-task")
for required in ("tasks/jumper/stairs/mdp/actions.py", "tools/stairs_tremor.py", "tools/stairs_record.py"):
    assert (REPO / required).is_file(), f"manca {required}: serve il Dataset v12 o successivo"

if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False).create(REPO / ".venv")
run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "-q", "--upgrade", "pip"])
run([PYTHON, "-m", "pip", "install", "-q", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cpu"])
run([PYTHON, "-m", "pip", "install", "-q", "-e", REPO])
run(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "libegl-mesa0", "libgl1-mesa-dri", "libgles2", "ffmpeg"], check=False)
os.environ.update(MUJOCO_GL="egl", EGL_PLATFORM="surfaceless", PYOPENGL_PLATFORM="egl", PYTHONUNBUFFERED="1")
run([PYTHON, "-c", "import torch, mujoco; print('torch', torch.__version__, 'mujoco', mujoco.__version__, 'cuda', torch.cuda.is_available())"])
listing = run([PYTHON, "scripts/train.py", "--list"], cwd=REPO).stdout
assert TASK in listing, f"{TASK} non registrata"
print("SETUP OK")


In [ ]:
# @title 2. Il backup migliore di C dall'Output della sessione 2 (versione 9)
archives = sorted(glob.glob("/kaggle/input/**/backups/jumper.stairs_free-*.zip", recursive=True))
assert archives, "Add Input: l'output del notebook jumper-kaggle-stairs (versione 9)"
ARCHIVE = archives[-1]
print("Backup:", ARCHIVE)
info = WORK / "restore.json"
run([PYTHON, "tools/colab.py", "restore", "--archive", ARCHIVE, "--destination", WORK / "restored-c", "--out", info], cwd=REPO)
CHECKPOINT = Path(json.loads(info.read_text())["checkpoint"])
print("Checkpoint:", CHECKPOINT)


In [ ]:
# @title 3. Gradino singolo: salto e tremolio, rumore acceso/spento x filtro (CPU, ~15-20 min)
rows_file = OUT / "tremor.json"
run([PYTHON, "tools/stairs_tremor.py", "--task", TASK, "--checkpoint", CHECKPOINT, "--row", ROW, "--layout", LAYOUT,
     "--num-envs", NUM_ENVS, "--seconds", SECONDS, "--alphas", ALPHAS, "--noise", "on,off",
     "--out", rows_file], cwd=REPO)
rows = json.loads(rows_file.read_text())
print("\nRISULTATI (riga", ROW, ", layout", LAYOUT, ")")
print(f"{'rumore':<7}{'alpha':>6}{'sale':>7}{'cade':>7}{'apice cm':>10}{'qd_rms':>9}{'dtarget mrad':>14}{'>5Hz':>7}")
for r in rows:
    print(f"{'on' if r['noise'] else 'off':<7}{r['alpha']:>6g}{r['climbed']:>7.2f}{r['fell']:>7.2f}{r['apex_cm']:>10.1f}"
          f"{(r['qd_rms'] or float('nan')):>9.3f}{(r['dtarget_rms_mrad'] or float('nan')):>14.2f}{(r['hf_share'] or float('nan')):>7.2f}")


In [ ]:
# @title 4. Registra i salti migliori (traiettorie per il render)
TRAJ = OUT / "trajectories"
for alpha in RECORD_ALPHAS:
    folder = TRAJ / f"curb15_alpha{alpha:g}"
    run([PYTHON, "tools/stairs_record.py", "--task", TASK, "--checkpoint", CHECKPOINT, "--row", ROW,
         "--layout", LAYOUT, "--alpha", alpha, "--noise", "on", "--num-envs", RECORD_ENVS,
         "--keep", RECORD_KEEP, "--seconds", SECONDS, "--out", folder], cwd=REPO)
shutil.make_archive(str(OUT / "trajectories"), "zip", TRAJ)
print("ZIP", OUT / "trajectories.zip", (OUT / "trajectories.zip").stat().st_size // 1024, "KB")


In [ ]:
# @title 5. Video di controllo sul gradino singolo, camera laterale (CPU, rendering Mesa)
from IPython.display import Video, display
rates_file = WORK / "rates.json"
run([PYTHON, "tools/colab.py", "rates", "--repo", REPO, "--task", TASK, "--model", "jumper", "--out", rates_file], cwd=REPO)
rates = json.loads(rates_file.read_text())
steps = int(round(VIDEO_SECONDS * rates["control_hz"]))
for label, alpha, noise in VIDEOS:
    video = OUT / f"{label}-row{ROW}.mp4"
    cmd = [PYTHON, "scripts/play.py", "--task", TASK, "--model", "jumper", "--checkpoint", CHECKPOINT,
           "--backend", "native", "--device", "cpu", "--num_envs", "1", "--headless", "--steps", steps,
           "--physics-hz", rates["physics_hz"], "--video", video, "--video-fps", "30",
           "--video-width", "960", "--video-height", "540", "--strip-visual", "off",
           "--video-azimuth", "90", "--video-elevation", "-8", "--video-distance", "1.3",
           "--stairs-row", ROW, "--stairs-layout", LAYOUT, "--action-ema", alpha, "--obs-noise" if noise else "--no-obs-noise"]
    try:
        run(cmd, cwd=REPO)
        print("VIDEO", video.name)
        display(Video(str(video), embed=True))
    except Exception as exc:  # noqa: BLE001
        print(f"[video {label}] fallito: {exc}")
print("FATTO: tabella in curb/tremor.json, traiettorie in curb/trajectories.zip, video in curb/")
